In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import uproot
import awkward as ak
import pandas as pd
from scipy.optimize import curve_fit
import sys
import pickle
import os

In [ ]:
# Run number is given as an argument
# run_number = int(sys.argv[1])
run_number = 19520
file_path = "/raid1/genli/Data_D2O/run" + str(run_number) + "_processed_v5.root"
Bithist_path = '/raid1/genli/Data_D2O/Bit_hist'
branches_to_read = ["eventID", "nsTime", "triggerBits", "pulseH", "area"]

# Open the ROOT file and access the tree
with uproot.open(file_path) as file:
    tree = file["tree"]
    
    # Initialize an empty list to store the processed data
    data_list = []
    
    # Process the tree in chunks to reduce memory usage
    for chunk in tree.iterate(branches_to_read, library="ak", step_size="100 MB"):
        # Convert scalar branches to a dictionary
        scalar_data = {key: ak.to_numpy(chunk[key]) for key in ["eventID", "nsTime", "triggerBits"]}
        
        # Convert multi-dimensional arrays (e.g., pulseH and area) and store as lists of arrays
        array_data = {key: list(ak.to_numpy(chunk[key])) for key in ["pulseH", "area"]}
        
        # Calculate sum_area using vectorized operations
        area_arrays = ak.to_numpy(chunk["area"])
        scalar_data['sum_area'] = np.sum(area_arrays[:, 0:11], axis=1)
        
        # Combine scalar and array data into a single dictionary
        combined_data = {**scalar_data, **array_data}
        
        # Append the processed chunk to the list
        data_list.append(pd.DataFrame(combined_data))
    
    # Concatenate all chunks into a single DataFrame
    df_new = pd.concat(data_list, ignore_index=True)

# Save the DataFrame to a pickle file
output_path = "/raid1/genli/Data_D2O/run" + str(run_number) + "_data_new.pkl"
df_new.to_pickle(output_path)

In [ ]:
def save_trigger_bits_histogram(df, img_path, pkl_path):
    """
    Generates and saves a histogram of the specified column in log scale.
    Saves the histogram figure as a PNG and the histogram data (counts and bins) as a pickle file.
    
    Parameters:
        df (DataFrame): Input DataFrame.
        img_path (str): File path to save the histogram image.
        pkl_path (str): File path to save the histogram data as a pickle file.
    """
    # Extract data
    trigger_bits = df['triggerBits'].to_numpy()
    
    # Define bins
    bins_array = np.linspace(0, 35, 36)
    
    # Create figure
    plt.figure(figsize=(8, 5))
    
    # Compute histogram data
    hist, bins, _ = plt.hist(trigger_bits, bins=bins_array, edgecolor='black', alpha=0.7)
    
    # Plot formatting
    plt.xlabel("Trigger Bits")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.title("Histogram of Trigger Bits")
    
    plt.minorticks_on()
    plt.grid(which='major', axis='y', linestyle='-', linewidth=0.75, color='gray')
    plt.grid(which='minor', axis='y', linestyle=':', linewidth=0.5, color='gray')
    plt.grid(which='both', axis='x', linestyle='--', linewidth=0.5, color='gray')
    
    # Save figure
    plt.savefig(img_path)
    plt.show()
    
    # Save histogram data
    hist_data = {"hist": hist, "bins": bins}
    with open(pkl_path, "wb") as f:
        pickle.dump(hist_data, f)
    
    print(f"Trigger Bit Histogram image saved to: {img_path}")
    print(f"Trigger Bit Histogram data saved to: {pkl_path}")

In [ ]:
def save_histograms(df, column_name, trigger_column, trigger_value, img_path, pkl_path):
    """
    Generates and saves histograms of the specified column, comparing all data vs. a subset where the trigger condition is met.
    Saves the histogram figure as a JPEG and the histogram data as a pickle file.
    
    Parameters:
        df (DataFrame): Input DataFrame.
        column_name (str): The column name to create the histograms for.
        trigger_column (str): The column used to filter data based on a trigger value.
        trigger_value (int): The trigger value to filter by.
        img_path (str): File path to save the histogram image.
        pkl_path (str): File path to save the histogram data as a pickle file.
    """
    # Extract data
    all_sums = df[column_name].to_numpy()
    trigger_sums = df.loc[df[trigger_column] == trigger_value, column_name].to_numpy()
    
    # Generate random indices for downsampling
    idx_all = np.random.choice(len(all_sums), size=int(len(all_sums)), replace=False)
    idx_trigger = np.random.choice(len(trigger_sums), size=int(len(trigger_sums)), replace=False)
    
    # Downsample the data
    sample_all_sums = all_sums[idx_all]
    sample_trigger_sums = trigger_sums[idx_trigger]
    
    # Create figure and plot histograms
    plt.figure(figsize=(12, 8))
    bins_num = 100
    bins_p = np.linspace(0, 100000, bins_num)
    
    plt.hist(sample_all_sums, bins=bins_p, alpha=0.7, edgecolor='black', label='All triggerBits')
    plt.hist(sample_trigger_sums, bins=bins_p, alpha=0.7, edgecolor='black', label=f'{trigger_column} = {trigger_value}')
    
    plt.xlabel("Total Charge (ADC)", fontsize=25)
    plt.ylabel("Events", fontsize=25)
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)
    plt.title("Histogram of Total Charge (ADC)", fontsize=25)
    plt.yscale("log")
    plt.legend(fontsize=20)
    
    plt.minorticks_on()
    plt.grid(which='major', axis='y', linestyle='-', linewidth=0.75, color='gray')
    plt.grid(which='minor', axis='y', linestyle=':', linewidth=0.5, color='gray')
    plt.grid(which='both', axis='x', linestyle='--', linewidth=0.5, color='gray')
    
    plt.xlim(0, 100000)
    plt.tight_layout()
    
    # Save figure
    plt.savefig(img_path)
    plt.show()
    
    # Save histogram data
    hist_data = {"all_sums": sample_all_sums, "trigger_sums": sample_trigger_sums, "bins": bins_p}
    with open(pkl_path, "wb") as f:
        pickle.dump(hist_data, f)
    
    print(f"Energy Histogram image saved to: {img_path}")
    print(f"Energy Histogram data saved to: {pkl_path}")


In [ ]:
def generate_and_save_histograms(df, deltaT_cut, sum_area_cut, bins_num, save_folder):
    """
    Generate histograms for Δt and sum_area for after-veto events after applying
    both a Δt cut and an energy (sum_area) cut. Save both the histogram plots and 
    the histogram data (counts, bin centers, error bars) as pickle files.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame with at least the columns 'triggerBits',
                           'nsTime', and 'sum_area'.
        deltaT_cut (tuple): Two-element tuple (dt_min, dt_max) defining the Δt range (in ns).
        sum_area_cut (tuple): Two-element tuple (s_min, s_max) defining the sum_area range.
        bins_num (int): The number of bins to use in the histograms.
        save_folder (str): The folder path where the output plots and pickle files will be saved.
    """
    # Make sure the save folder exists.
    os.makedirs(save_folder, exist_ok=True)
    
    # --- Step 1: Identify muon and after-veto events, and compute Δt ---
    # Muon events: triggerBits >= 32.
    muon_mask = df['triggerBits'] >= 32
    # After-veto events: triggerBits == 2.
    after_veto_mask = df['triggerBits'] == 2

    # Extract muon times (assumes df is sorted by 'nsTime').
    muon_times = df.loc[muon_mask, 'nsTime'].values

    # Copy after-veto events to avoid modifying the original DataFrame.
    after_veto_df = df.loc[after_veto_mask].copy()
    event_times = after_veto_df['nsTime'].values

    # For each after-veto event, find the most recent muon event.
    insertion_indices = np.searchsorted(muon_times, event_times, side='right')
    delta_t = np.full_like(event_times, np.nan, dtype=float)
    valid = insertion_indices > 0
    delta_t[valid] = event_times[valid] - muon_times[insertion_indices[valid] - 1]
    after_veto_df['delta_t'] = delta_t

    # --- Step 2: Apply the cuts ---
    dt_min, dt_max = deltaT_cut
    s_min, s_max = sum_area_cut

    # Select events that satisfy both the Δt and sum_area cuts.
    selected = after_veto_df[
        (after_veto_df['delta_t'] >= dt_min) &
        (after_veto_df['delta_t'] <= dt_max) &
        (after_veto_df['sum_area'] >= s_min) &
        (after_veto_df['sum_area'] <= s_max)
    ]
    
    # Check if any events were selected
    if selected.empty:
        print("No events passed the selection cuts.")
        return

    # --- Step 3: Create and save the Δt histogram ---
    dt_bins = np.linspace(dt_min, dt_max, bins_num)
    dt_hist, dt_bin_edges = np.histogram(selected['delta_t'], bins=dt_bins)
    dt_bin_centers = (dt_bin_edges[:-1] + dt_bin_edges[1:]) / 2
    dt_errorbars = np.sqrt(dt_hist)  # Poisson errors

    # Save the histogram data as a pickle file.
    dt_data = {
        'hist': dt_hist,
        'bin_centers': dt_bin_centers,
        'errorbars': dt_errorbars
    }
    dt_pkl_path = os.path.join(save_folder, 'delta_t_histogram.pkl')
    with open(dt_pkl_path, 'wb') as f:
        pickle.dump(dt_data, f)

    # Plot the Δt histogram with error bars.
    plt.figure(figsize=(10, 7))
    plt.errorbar(dt_bin_centers, dt_hist, yerr=dt_errorbars, fmt='o', label='Data')
    plt.xlabel("Δt (ns)", fontsize=14)
    plt.ylabel("Counts", fontsize=14)
    plt.title("Δt Histogram (Selected Events)", fontsize=16)
    plt.legend()
    plt.tight_layout()
    dt_plot_path = os.path.join(save_folder, 'delta_t_histogram.png')
    plt.savefig(dt_plot_path)
    plt.close()

    # --- Step 4: Create and save the sum_area histogram ---
    s_bins = np.linspace(s_min, s_max, bins_num)
    s_hist, s_bin_edges = np.histogram(selected['sum_area'], bins=s_bins)
    s_bin_centers = (s_bin_edges[:-1] + s_bin_edges[1:]) / 2
    s_errorbars = np.sqrt(s_hist)  # Poisson errors

    # Save the histogram data as a pickle file.
    s_data = {
        'hist': s_hist,
        'bin_centers': s_bin_centers,
        'errorbars': s_errorbars
    }
    s_pkl_path = os.path.join(save_folder, 'sum_area_histogram.pkl')
    with open(s_pkl_path, 'wb') as f:
        pickle.dump(s_data, f)

    # Plot the sum_area histogram with error bars.
    plt.figure(figsize=(10, 7))
    plt.errorbar(s_bin_centers, s_hist, yerr=s_errorbars, fmt='o', label='Data')
    plt.xlabel("Total Charge (ADC)", fontsize=14)
    plt.ylabel("Counts", fontsize=14)
    plt.title("Total Charge (sum_area) Histogram (Selected Events)", fontsize=16)
    plt.legend()
    plt.tight_layout()
    s_plot_path = os.path.join(save_folder, 'sum_area_histogram.png')
    plt.savefig(s_plot_path)
    plt.close()

    print(f"delta T and Total_Charge Histograms and data files saved in: {save_folder}")